# 11 — Validação Final

Abre os blocos de teste **uma única vez**, depois de `G7`, `G8` e `G9` aprovados. Reconstrói o pipeline do YAML (treino em `train_ref`), calibra na metade `val_cal` do `val_ref` (mesmo split do notebook 10) e avalia o modelo congelado em três janelas: **Teste A** (`test_ref`, resultado principal), **2020–2022** (`test_pandemic`, isolado) e **Teste B** (`test_ref` + `test_pandemic`). Nada é reajustado.

No final, marca `G9_holdout` como `executado` e registra `test_opened` no manifesto do artefato.

**Ensaio sem abrir o teste:** com `SRAG_ENSAIO=1` o notebook roda de ponta a ponta usando duas metades da `val_meas` no lugar dos blocos de teste. Não lê o teste, não marca o G9, não altera o artefato e grava com o prefixo `11_ensaio_` (que não aciona a guarda de reabertura).

In [3]:
from pathlib import Path
import hashlib
import json
import os
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

try:
    import sklearn.frozen  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Este notebook requer scikit-learn >= 1.6 (FrozenEstimator)."
    ) from exc

from src import config as C
from src import config as env
from src.registro import Run
from src.dados import MANIFEST_NAME, GatePendenteError, load_supervised_artifact, split_partitions
from src.avaliacao import metricas_completas, ic_bootstrap, bootstrap_grupo, auc, average_precision, brier as brier_metric
from src.avaliacao import matriz_confusao, metricas_por_grupo
from src.avaliacao import curva_confiabilidade, ece, mce, brier_decomposicao
from src.features import faixa_etaria as _faixa_etaria
from src.modelos import build_finalista, finalistas_aceitos, resolver_finalista
from src.modelos import calibrar

HASH_LEN                = 16
CALIB_SPLIT_FRAC_PADRAO = 0.5
N_BINS_CALIB            = 10
MIN_N_CALIB_ANO         = 20
N_FN_AMOSTRA            = 20
TAMANHO_LOTE_PREDICAO   = 250_000
LIMITE_LINHAS_COM_FEATURES = 500_000

ENSAIO       = os.getenv("SRAG_ENSAIO", "0") == "1"
PREFIXO_BASE = "11_ensaio_validacao_final" if ENSAIO else "11_validacao_final"

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR      = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(C.resumo())
if ENSAIO:
    print("=" * 70)
    print("MODO ENSAIO: o teste NÃO será lido, o G9 NÃO será marcado e as saídas levam o prefixo 11_ensaio_.")
    print("=" * 70)

ROOT_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
DATA_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data
PROCESSED_DIR     = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\processed
ANALYTICAL_DIR    = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\analytical
OUTPUTS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs
RUNS_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs
CONFIGS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs
SUPERVISED_YAML   = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs\supervised.yaml (ok)
PARQUET_NAME      = srag.parquet
DATA_VERSION      = dados-v2
GITHUB_REPO       = Ivan-S-Oliveira/Projeto_Integrador
HTTP_TIMEOUT      = 180
HTTP_TENTATIVAS   = 3
SEED              = 42
GITHUB_TOKEN?     = False
SRAG_PARQUET_URL  = https://gi

In [4]:
GATES_EXIGIDOS = ["G7_calibracao", "G8_limiar", "G9_holdout"]
status_gates   = {g: (C.cfg("gates", g) or {}).get("status") for g in GATES_EXIGIDOS}

display(pd.DataFrame({"gate": list(status_gates), "status": list(status_gates.values())}))

n_acessos_antes = int((C.cfg("gates", "G9_holdout") or {}).get("n_acessos") or 0)

if ENSAIO:
    print("ENSAIO: gates não exigidos (o teste não é aberto).")
else:
    if faltando := [g for g, s in status_gates.items() if s != "aprovado"]:
        raise RuntimeError(
            f"Gates não aprovados: {faltando}. Edite configs/supervised.yaml."
        )
    print(f"Gates aprovados. n_acessos_antes (G9) = {n_acessos_antes}.")

,gate,status
0,G7_calibracao,aprovado
1,G8_limiar,aprovado
2,G9_holdout,aprovado


Gates aprovados. n_acessos_antes (G9) = 1.


In [5]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "final").lower()

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
    partitions=("train_ref", "val_ref"),
)

PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

FEATURES_YAML = list(C.cfg("features", "final") or [])
if not FEATURES_YAML:
    raise RuntimeError("features.final vazio — congele as features antes do notebook 11.")
if set(FEATURES) != set(FEATURES_YAML):
    raise RuntimeError(
        "features.final do YAML difere das variáveis do artefato "
        f"(só no YAML: {sorted(set(FEATURES_YAML) - set(FEATURES))}; "
        f"só no artefato: {sorted(set(FEATURES) - set(FEATURES_YAML))}). "
        "Regere o artefato (notebook 02) e rode o notebook 10 de novo."
    )

selecao_cfg = C.cfg("selecao_modelos") or {}
finalista_escolhido = selecao_cfg.get("finalista_escolhido")
if finalista_escolhido is None:
    candidatos = list(selecao_cfg.get("finalistas") or [])
    if ENSAIO and candidatos:
        finalista_escolhido = "logistic" if "logistic" in candidatos else candidatos[0]
        print(f"ENSAIO: finalista_escolhido vazio no YAML; usando {finalista_escolhido!r}.")
    elif len(candidatos) == 1:
        finalista_escolhido = candidatos[0]
    else:
        raise RuntimeError(
            "selecao_modelos.finalista_escolhido não definido e "
            f"finalistas={candidatos}. Congele o finalista no YAML."
        )

calib_cfg    = C.cfg("calibracao") or {}
calib_ativa  = bool(calib_cfg.get("ativa", True))
calib_metodo = calib_cfg.get("metodo")
calib_fonte  = str(calib_cfg.get("fonte") or "validacao")

if calib_ativa and calib_metodo is None:
    if ENSAIO:
        calib_metodo = "sigmoid"
        print("ENSAIO: calibracao.metodo vazio no YAML; usando 'sigmoid'.")
    else:
        raise RuntimeError("calibracao.ativa=true mas calibracao.metodo vazio no YAML.")
if calib_fonte != "validacao":
    raise RuntimeError(f"calibracao.fonte deve ser 'validacao', não {calib_fonte!r}.")

limiar_cfg     = C.cfg("limiar") or {}
limiar_final   = limiar_cfg.get("otimo")
limiar_metrica = limiar_cfg.get("metrica")
if limiar_final is None:
    if ENSAIO:
        limiar_final = 0.5
        print("ENSAIO: limiar.otimo vazio no YAML; usando 0,5.")
    else:
        raise RuntimeError("limiar.otimo vazio no YAML.")
limiar_final_val = float(limiar_final)

proibidas = set(C.cfg("features", "proibidas") or [])
if conflito := set(FEATURES) & proibidas:
    raise RuntimeError(f"features.final contém colunas proibidas: {sorted(conflito)}")

janelas_cfg = C.cfg("avaliacao", "janelas") or {}
JANELAS = {nome: list((cfg or {}).get("blocos") or []) for nome, cfg in janelas_cfg.items()}
if not {"teste_a", "teste_b", "pandemico_isolado"} <= set(JANELAS):
    raise RuntimeError(f"avaliacao.janelas incompleto no YAML: {sorted(JANELAS)}.")
JANELA_PRINCIPAL = next(
    (n for n, cfg in janelas_cfg.items() if (cfg or {}).get("resultado_principal")), "teste_a"
)
BLOCOS_TESTE = list(dict.fromkeys(b for blocos in JANELAS.values() for b in blocos))

display(pd.DataFrame([
    {"chave": "finalista",         "valor": finalista_escolhido},
    {"chave": "calibracao.metodo", "valor": calib_metodo if calib_ativa else "desativada"},
    {"chave": "limiar.otimo",      "valor": limiar_final_val},
    {"chave": "limiar.metrica",    "valor": limiar_metrica},
    {"chave": "features.final",    "valor": len(FEATURES)},
    {"chave": "janelas",           "valor": {n: "+".join(b) for n, b in JANELAS.items()}},
    {"chave": "n_acessos_G9",      "valor": n_acessos_antes},
    {"chave": "ensaio",            "valor": ENSAIO},
]))

,chave,valor
0,finalista,boosting
1,calibracao.metodo,sigmoid
2,limiar.otimo,0.1178
3,limiar.metrica,precision
4,features.final,30
5,janelas,"{'teste_a': 'test_ref', 'teste_b': 'test_ref+t..."
6,n_acessos_G9,1
7,ensaio,False


In [6]:
ALLOW_REOPEN = os.getenv("SRAG_ALLOW_TEST_REOPEN", "0") == "1"

reaberturas = []
if ENSAIO:
    print("ENSAIO: a guarda de reabertura não se aplica (o teste não é aberto).")
else:
    for caminho in TABLES_DIR.glob("11_validacao_final_*_manifesto.json"):
        try:
            dados = json.loads(caminho.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if dados.get("dataset_sha256") == manifest["dataset_sha256"]:
            reaberturas.append({"arquivo": caminho.name, "em_utc": dados.get("teste_aberto_em_utc")})
    if manifest.get("test_opened"):
        reaberturas.append({"arquivo": "manifesto_supervisionado.json", "em_utc": manifest.get("test_opened_utc")})

    if reaberturas and not ALLOW_REOPEN:
        display(pd.DataFrame(reaberturas))
        raise RuntimeError(
            "Teste já aberto para este dataset_sha256. Defina "
            "SRAG_ALLOW_TEST_REOPEN=1 para reabrir e registre o motivo em docs/decisoes.md."
        )
    print("Guarda OK." if not reaberturas else f"Reabertura autorizada ({len(reaberturas)} registro(s) anterior(es)).")

Guarda OK.


In [7]:
def _reproduzir_split_val_cal(y_val, frac_cal, seed):
    idx_cal, idx_meas = train_test_split(
        np.arange(len(y_val)),
        test_size=1.0 - frac_cal,
        random_state=seed,
        stratify=y_val.to_numpy(),
    )
    return np.asarray(idx_cal, dtype=np.int64), np.asarray(idx_meas, dtype=np.int64)


def _chave_c5(finalista):
    """Nome do finalista como o notebook 10 o grava em `escolhas` (o que está em selecao_modelos.finalistas)."""
    for nome in (selecao_cfg.get("finalistas") or []):
        if resolver_finalista(nome) == resolver_finalista(finalista):
            return nome
    return finalista


def _localizar_manifesto_c5(dataset_sha256, finalista, metodo, calib_ativa):
    for caminho in sorted(
        TABLES_DIR.glob("10_calibracao_limiar_*_manifesto.json"),
        key=lambda p: p.stat().st_mtime, reverse=True,
    ):
        try:
            dados = json.loads(caminho.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if dados.get("dataset_sha256") != dataset_sha256:
            continue
        if calib_ativa:
            escolha = (dados.get("escolhas") or {}).get(_chave_c5(finalista)) or {}
            if escolha.get("metodo") != metodo:
                continue
        return dados, caminho
    return None, None


def _hash_config_modelo(features, finalista):
    carga = {"models": C.cfg("models"), "features": sorted(features), "finalista": resolver_finalista(finalista)}
    return hashlib.sha256(json.dumps(carga, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:HASH_LEN]


treino, validacao = split_partitions(df, manifest)

X_train = treino[FEATURES]
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")
X_val   = validacao[FEATURES]
y_val   = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

manifesto_c5, manifesto_c5_path = _localizar_manifesto_c5(
    manifest["dataset_sha256"], finalista_escolhido,
    calib_metodo if calib_ativa else None, calib_ativa,
)

if manifesto_c5 is None:
    if ENSAIO:
        print("ENSAIO: manifesto do notebook 10 não encontrado; usando o split padrão (honesto, 50%, semente do YAML).")
    elif calib_ativa:
        raise RuntimeError(
            "Nenhum manifesto notebook 10 compatível (dataset_sha256 + metodo). "
            "Rode o notebook 10 com a configuração atual do YAML antes do notebook 11."
        )

if manifesto_c5 is not None and not ENSAIO:
    if set(manifesto_c5.get("feature_columns") or []) != set(FEATURES):
        raise RuntimeError("O notebook 10 foi rodado com outras variáveis: rode o notebook 10 de novo antes do notebook 11.")
    limiar_c5 = ((manifesto_c5.get("escolhas") or {}).get(_chave_c5(finalista_escolhido)) or {}).get("limiar")
    if limiar_c5 is None or abs(float(limiar_c5) - limiar_final_val) > 1e-6:
        raise RuntimeError(
            f"O limiar do manifesto do notebook 10 ({limiar_c5}) não é o do YAML ({limiar_final_val}): "
            "copie o limiar impresso pelo notebook 10 para limiar.otimo ou rode o notebook 10 de novo."
        )
    hash_modelo_c5 = manifesto_c5.get("config_modelo_hash")
    hash_modelo_agora = _hash_config_modelo(FEATURES, finalista_escolhido)
    if hash_modelo_c5 != hash_modelo_agora:
        raise RuntimeError(
            "Os modelos do YAML mudaram depois do notebook 10 (ou o notebook 10 é de uma versão sem registro do hash): "
            f"notebook 10={hash_modelo_c5}, agora={hash_modelo_agora}. Rode o notebook 10 de novo antes do notebook 11."
        )

HONEST_CALIB_SPLIT = bool(manifesto_c5.get("honest_calib_split", False)) if manifesto_c5 else True
CALIB_SPLIT_FRAC   = float(manifesto_c5.get("calib_split_frac", CALIB_SPLIT_FRAC_PADRAO)) if manifesto_c5 else CALIB_SPLIT_FRAC_PADRAO
CALIB_SPLIT_SEED   = int(manifesto_c5.get("calib_split_seed", C.SEED)) if manifesto_c5 else int(C.SEED)

X_val_cal, y_val_cal = X_val, y_val
validacao_meas       = validacao
if HONEST_CALIB_SPLIT:
    idx_cal, idx_meas = _reproduzir_split_val_cal(y_val, CALIB_SPLIT_FRAC, CALIB_SPLIT_SEED)
    hash_local = hashlib.sha256(idx_cal.tobytes()).hexdigest()[:HASH_LEN]
    hash_c5    = manifesto_c5.get("calib_split_hash") if manifesto_c5 else None
    if hash_c5 and hash_local != hash_c5:
        raise RuntimeError(f"Split divergente do notebook 10 (local={hash_local}, notebook 10={hash_c5}).")
    X_val_cal = X_val.iloc[idx_cal].reset_index(drop=True)
    y_val_cal = y_val.iloc[idx_cal].reset_index(drop=True)
    validacao_meas = validacao.iloc[idx_meas].reset_index(drop=True)

if finalista_escolhido not in finalistas_aceitos():
    raise ValueError(
        f"Finalista desconhecido: {finalista_escolhido!r}. "
        f"Aceitos: {list(finalistas_aceitos())}."
    )

pipe_base = build_finalista(finalista_escolhido)
pipe_base.fit(X_train, y_train)

if calib_ativa:
    if calib_metodo is None:
        raise ValueError("calib_metodo deve ser definido quando a calibração está ativa.")
    calibrador = calibrar(pipe_base, X_val_cal, y_val_cal, method=calib_metodo)
else:
    calibrador = None

preditor = calibrador if calibrador is not None else pipe_base

display(pd.DataFrame({
    "bloco":       ["treino (train_ref)", "calibração (val_cal)", "medição (val_meas, não usada aqui)"],
    "n":           [len(treino), len(X_val_cal), len(validacao_meas)],
    "positivos":   [int(y_train.sum()), int(y_val_cal.sum()), int(pd.to_numeric(validacao_meas[ALVO]).sum())],
    "prevalencia": [float(y_train.mean()), float(y_val_cal.mean()), float(pd.to_numeric(validacao_meas[ALVO]).mean())],
}))
print(f"finalista   : {finalista_escolhido} (fábrica: {resolver_finalista(finalista_escolhido)})")
print(f"calibração  : {calib_metodo if calib_ativa else 'desativada'}")
print(f"split notebook 10: {'honesto' if HONEST_CALIB_SPLIT else 'validação inteira'}")

,bloco,n,positivos,prevalencia
0,treino (train_ref),556462,46961,0.084392
1,calibração (val_cal),59621,5032,0.084400
2,"medição (val_meas, não usada aqui)",59622,5032,0.084398


finalista   : boosting (fábrica: boosting)
calibração  : sigmoid
split notebook 10: honesto


In [8]:
def prever_em_lotes(modelo, X, tamanho=TAMANHO_LOTE_PREDICAO):
    partes = [modelo.predict_proba(X.iloc[i:i + tamanho])[:, 1] for i in range(0, len(X), tamanho)]
    return np.concatenate(partes) if partes else np.empty(0, dtype="float64")


COLUNA_ANO = str(C.cfg("periodo", "coluna_ano") or "ano_internacao")
_audit_manifesto = set(manifest.get("audit_columns") or [])
COLUNAS_AUDITORIA = [
    c for c in ("id_registro", "group_id", COLUNA_ANO, "sg_uf", "regiao_residencia", "faixa_etaria", "sexo_categoria")
    if c in _audit_manifesto or c in FEATURES
]

abertura_teste_utc = pd.Timestamp.now(tz="UTC").isoformat()
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo   = f"{PREFIXO_BASE}_{manifest['cohort_kind']}_{timestamp}"

blocos_teste = {}
if ENSAIO:
    meio = len(validacao_meas) // 2
    partes_ensaio = {
        "test_ref": validacao_meas.iloc[:meio].reset_index(drop=True),
        "test_pandemic": validacao_meas.iloc[meio:].reset_index(drop=True),
    }
    fonte_blocos = lambda bloco: partes_ensaio[bloco]  # noqa: E731
else:
    colunas_leitura = list(dict.fromkeys(FEATURES + COLUNAS_AUDITORIA + [ALVO]))

    def fonte_blocos(bloco):
        try:
            parte, _ = load_supervised_artifact(
                ROOT_DIR, expected_kind=COHORT_KIND, partitions=(bloco,), columns=colunas_leitura,
            )
        except GatePendenteError as exc:
            raise RuntimeError(
                "G9_holdout ficou pendente entre a checagem e o carregamento. "
                "Verifique configs/supervised.yaml antes de rodar novamente."
            ) from exc
        return parte.reset_index(drop=True)

for bloco in BLOCOS_TESTE:
    parte = fonte_blocos(bloco)
    y_bloco = pd.to_numeric(parte[ALVO], errors="coerce").astype("int8").to_numpy()
    prob_bloco = prever_em_lotes(preditor, parte[FEATURES])
    if not np.isfinite(prob_bloco).all() or not ((prob_bloco >= 0) & (prob_bloco <= 1)).all():
        raise RuntimeError(f"Probabilidades inválidas no bloco {bloco}.")
    colunas_guardar = COLUNAS_AUDITORIA + ([c for c in FEATURES if c not in COLUNAS_AUDITORIA] if len(parte) <= LIMITE_LINHAS_COM_FEATURES else [])
    meta = parte[[c for c in colunas_guardar if c in parte.columns]].copy()
    blocos_teste[bloco] = {"meta": meta, "y": y_bloco, "prob": prob_bloco}
    del parte


def montar_janela(blocos, com_features=False):
    partes = [blocos_teste[b] for b in blocos]
    colunas = COLUNAS_AUDITORIA + ([c for c in FEATURES if c not in COLUNAS_AUDITORIA] if com_features else [])
    meta = pd.concat(
        [p["meta"][[c for c in colunas if c in p["meta"].columns]].assign(bloco=b) for p, b in zip(partes, blocos)],
        ignore_index=True,
    )
    return {
        "blocos": list(blocos),
        "meta": meta,
        "y": np.concatenate([p["y"] for p in partes]),
        "prob": np.concatenate([p["prob"] for p in partes]),
    }


JANELAS_DADOS = {nome: montar_janela(blocos, com_features=(nome == JANELA_PRINCIPAL)) for nome, blocos in JANELAS.items()}

linhas_janelas = []
for nome, j in JANELAS_DADOS.items():
    m = metricas_completas(j["y"], j["prob"], limiar=limiar_final_val)
    prevalencia = float(j["y"].mean())
    ap_janela = float(m.get("pr_auc", m.get("ap")))
    linhas_janelas.append({
        "janela": nome, "blocos": "+".join(j["blocos"]), "n": int(len(j["y"])),
        "positivos": int(j["y"].sum()), "prevalencia": prevalencia,
        "pr_auc_sobre_prevalencia": ap_janela / prevalencia if prevalencia > 0 else float("nan"),
        **{k: float(v) for k, v in m.items() if k != "n"},
    })
janelas_df = pd.DataFrame(linhas_janelas)

principal   = JANELAS_DADOS[JANELA_PRINCIPAL]
y_test_arr  = principal["y"]
prob_test   = principal["prob"]
y_pred_test = (prob_test >= limiar_final_val).astype(int)
metricas_test = metricas_completas(y_test_arr, prob_test, limiar=limiar_final_val)
matriz_test   = matriz_confusao(y_test_arr, y_pred_test)

display(janelas_df)
display(matriz_test)

def _preparar_para_arquivo(tabela):
    for coluna in tabela.columns:
        if str(tabela[coluna].dtype) in ("object", "category"):
            tabela[coluna] = tabela[coluna].astype("string")
    return tabela


pred_teste = pd.concat(
    [
        pd.DataFrame({
            "bloco": b, "y_true": blocos_teste[b]["y"], "y_prob": blocos_teste[b]["prob"],
            "y_pred": (blocos_teste[b]["prob"] >= limiar_final_val).astype("int8"),
        }).join(blocos_teste[b]["meta"][[c for c in COLUNAS_AUDITORIA if c in blocos_teste[b]["meta"].columns]])
        for b in BLOCOS_TESTE
    ],
    ignore_index=True,
)
pred_teste["limiar"] = limiar_final_val
pred_teste["modelo"] = finalista_escolhido
pred_teste["cohort_kind"] = manifest["cohort_kind"]
pred_teste = _preparar_para_arquivo(pred_teste)

caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_teste.parquet"
try:
    pred_teste.to_parquet(caminho_predicoes, index=False)
except Exception as exc:  # noqa: BLE001
    caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_teste.csv.gz"
    pred_teste.to_csv(caminho_predicoes, index=False, compression="gzip")
    print(f"AVISO: não consegui gravar parquet ({exc!r}); previsões gravadas em {caminho_predicoes.name}.")
del pred_teste

print(f"Teste aberto em UTC: {abertura_teste_utc}")
for nome, j in JANELAS_DADOS.items():
    print(f"  {nome:<18} n={len(j['y']):>10,}  óbitos={int(j['y'].sum()):>9,}")
print(f"Previsões: {caminho_predicoes}")

,janela,blocos,n,positivos,prevalencia,pr_auc_sobre_prevalencia,auc,ap,pr_auc,brier,log_loss,f1,precision,recall,sensibilidade,especificidade,accuracy,limiar
0,teste_a,test_ref,119245,10065,0.084406,4.371167,0.868597,0.368953,0.368953,0.062754,0.210703,0.384189,0.251514,0.813115,0.813115,0.776928,0.779982,0.1178
1,teste_b,test_ref+test_pandemic,3035074,781160,0.257378,1.765835,0.734950,0.454486,0.454486,0.178759,0.536922,0.510329,0.376441,0.792030,0.792030,0.545300,0.608803,0.1178
2,pandemico_isolado,test_pandemic,2915829,771095,0.264451,1.726346,0.727977,0.456534,0.456534,0.183503,0.550263,0.512585,0.378964,0.791755,0.791755,0.533509,0.601802,0.1178


,pred_0,pred_1,total
real_0,84825,24355,109180
real_1,1881,8184,10065
total,86706,32539,119245


Teste aberto em UTC: 2026-11-07T00:32:00.678246+00:00
  teste_a            n=   119,245  óbitos=   10,065
  teste_b            n= 3,035,074  óbitos=  781,160
  pandemico_isolado  n= 2,915,829  óbitos=  771,095
Previsões: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\predictions\11_validacao_final_final_20261107T003200Z_predicoes_teste.parquet


In [9]:
boot_cfg   = C.cfg("avaliacao", "bootstrap") or {}
BOOT_ATIVO = bool(boot_cfg.get("ativo", True))
BOOT_REPS  = int(boot_cfg.get("repeticoes") or 1000)
BOOT_NIVEL = float(boot_cfg.get("nivel") or 0.95)
BOOT_SEED  = int(boot_cfg.get("seed") or C.SEED)
if ENSAIO:
    BOOT_REPS = min(BOOT_REPS, 50)

grupos_boot = principal["meta"]["group_id"].to_numpy() if "group_id" in principal["meta"].columns else None
IC_METODO   = "bootstrap_por_grupo" if grupos_boot is not None else "bootstrap_por_ficha"

ic_resultados = {}
if BOOT_ATIVO:
    for nome, fn in [("ap", average_precision), ("auc", auc), ("brier", brier_metric)]:
        if grupos_boot is not None:
            r_ic = bootstrap_grupo(
                y_test_arr, prob_test, grupos_boot,
                metrica=nome, n=BOOT_REPS, alpha=1.0 - BOOT_NIVEL, seed=BOOT_SEED,
            )
            est, inf, sup = r_ic["estimativa"], r_ic["ic_inferior"], r_ic["ic_superior"]
        else:
            est, inf, sup = ic_bootstrap(
                y_test_arr, prob_test, metrica=fn,
                n_boot=BOOT_REPS, alpha=1.0 - BOOT_NIVEL, seed=BOOT_SEED,
            )
        ic_resultados[nome] = {
            "estimativa": est, "ic_inferior": inf, "ic_superior": sup,
            "nivel": BOOT_NIVEL, "repeticoes": BOOT_REPS, "metodo": IC_METODO, "janela": JANELA_PRINCIPAL,
        }

ic_df = pd.DataFrame([{"metrica": k, **v} for k, v in ic_resultados.items()])
display(ic_df)

,metrica,estimativa,ic_inferior,ic_superior,nivel,repeticoes,metodo,janela
0,ap,0.368953,0.359114,0.378756,0.95,1000,bootstrap_por_grupo,teste_a
1,auc,0.868597,0.865440,0.871589,0.95,1000,bootstrap_por_grupo,teste_a
2,brier,0.062754,0.061833,0.063696,0.95,1000,bootstrap_por_grupo,teste_a


In [10]:
erros_cfg   = C.cfg("avaliacao", "analise_erros") or {}
ERROS_ATIVA = bool(erros_cfg.get("ativa", True))

df_err = principal["meta"].copy()
df_err["y_true"] = y_test_arr
df_err["y_pred"] = y_pred_test
df_err["y_prob"] = prob_test

fn = df_err[(df_err.y_true == 1) & (df_err.y_pred == 0)]
tp = df_err[(df_err.y_true == 1) & (df_err.y_pred == 1)]
fp = df_err[(df_err.y_true == 0) & (df_err.y_pred == 1)]
tn = df_err[(df_err.y_true == 0) & (df_err.y_pred == 0)]

display(pd.DataFrame([
    {"classe": "FN (óbito→cura)",  "n": len(fn), "frac": len(fn) / len(df_err)},
    {"classe": "TP (óbito→óbito)", "n": len(tp), "frac": len(tp) / len(df_err)},
    {"classe": "FP (cura→óbito)",  "n": len(fp), "frac": len(fp) / len(df_err)},
    {"classe": "TN (cura→cura)",   "n": len(tn), "frac": len(tn) / len(df_err)},
]))

comparacao_fn_tp = pd.DataFrame()
amostra_fn       = pd.DataFrame()
FEATURES_NO_TESTE = [c for c in FEATURES if c in df_err.columns]

if ERROS_ATIVA and len(fn) > 0 and len(tp) > 0:
    feats_num = [c for c in FEATURES_NO_TESTE if pd.api.types.is_numeric_dtype(df_err[c])]
    comparacao_fn_tp = pd.DataFrame([
        {"feature":    c,
         "fn_media":   float(fn[c].mean()),
         "tp_media":   float(tp[c].mean()),
         "delta_media": float(fn[c].mean() - tp[c].mean())}
        for c in feats_num
    ])
    if not comparacao_fn_tp.empty:
        comparacao_fn_tp["abs_delta"] = comparacao_fn_tp.delta_media.abs()
        comparacao_fn_tp = comparacao_fn_tp.sort_values("abs_delta", ascending=False).reset_index(drop=True)
        print("Features numéricas com maior diferença FN vs TP:")
        display(comparacao_fn_tp)

if len(fn) > 0:
    colunas_fn = FEATURES_NO_TESTE + [c for c in ("group_id", COLUNA_ANO, "regiao_residencia") if c in df_err.columns and c not in FEATURES_NO_TESTE]
    amostra_fn = fn.sort_values("y_prob", ascending=False).head(N_FN_AMOSTRA)[colunas_fn + ["y_prob"]]
    print(f"Top {N_FN_AMOSTRA} falsos negativos por probabilidade predita:")
    display(amostra_fn)

fn_por_grupo = {}
for grupo in ("regiao_residencia", "faixa_etaria"):
    if grupo in df_err.columns:
        tab = (
            df_err.assign(fn=((df_err.y_true == 1) & (df_err.y_pred == 0)).astype(int), obito=(df_err.y_true == 1).astype(int))
            .groupby(grupo, dropna=False)[["fn", "obito"]].sum().reset_index()
        )
        tab["pct_obitos_sem_alerta"] = tab["fn"] / tab["obito"].where(tab["obito"] > 0)
        fn_por_grupo[grupo] = tab
        display(tab)

,classe,n,frac
0,FN (óbito→cura),1881,0.015774
1,TP (óbito→óbito),8184,0.068632
2,FP (cura→óbito),24355,0.204243
3,TN (cura→cura),84825,0.711351


Features numéricas com maior diferença FN vs TP:


,feature,fn_media,tp_media,delta_media,abs_delta
0,idade_anos,35.166412,71.259949,-36.093536,36.093536
1,dias_sintomas_ate_internacao,3.835194,3.101662,0.733532,0.733532


Top 20 falsos negativos por probabilidade predita:


,idade_anos,dias_sintomas_ate_internacao,sexo_categoria,cs_gestant,cs_raca,cs_zona,ave_suino,febre,tosse,garganta,...,neurologic,pneumopati,imunodepre,renal,obesidade,out_morbi,group_id,ano_internacao,regiao_residencia,y_prob
104531,0.083333,0.0,feminino,6,4,1,2,1,9,9,...,2,2,2,2,2,1,31691611950798,2023,SUDESTE,0.117769
13091,55.000000,2.0,masculino,6,1,1,2,1,1,1,...,2,2,1,2,2,1,31748317267911,2025,SUDESTE,0.117748
36779,78.000000,1.0,feminino,5,1,1,2,2,1,2,...,2,2,2,2,2,1,31746628151543,2025,SUDESTE,0.117500
79944,50.000000,14.0,masculino,6,4,1,9,1,1,2,...,2,2,1,2,2,9,31698694207100,2023,SUDESTE,0.117483
28358,68.000000,3.0,feminino,5,4,1,2,1,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,31740654720121,2025,NORDESTE,0.117473
27396,73.000000,0.0,feminino,5,1,1,2,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,31764787328776,2025,SUDESTE,0.117420
85264,54.000000,0.0,feminino,5,1,1,2,NaN,1,NaN,...,NaN,NaN,NaN,NaN,NaN,1,31699616486991,2023,SUDESTE,0.117386
56090,65.000000,1.0,feminino,5,4,1,2,2,2,1,...,2,1,2,2,2,2,31709047039297,2024,SUDESTE,0.117265
83168,67.000000,5.0,feminino,5,4,1,2,2,1,2,...,2,1,2,2,1,1,31688988252330,2023,SUDESTE,0.117260
100809,89.000000,3.0,feminino,6,4,1,9,2,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,31697053002782,2023,SUDESTE,0.117234


,regiao_residencia,fn,obito,pct_obitos_sem_alerta
0,NORTE,137,491,0.279022
1,NORDESTE,311,1301,0.239047
2,CENTRO-OESTE,204,970,0.210309
3,SUDESTE,885,5208,0.169931
4,SUL,344,2091,0.164515
5,AUSENTE,0,4,0.000000


,faixa_etaria,fn,obito,pct_obitos_sem_alerta
0,1-4,222,233,0.952790
1,12-17,62,83,0.746988
2,18-39,183,559,0.327370
3,40-59,290,1562,0.185659
4,5-11,130,138,0.942029
5,60-79,421,3858,0.109124
6,80+,168,3196,0.052566
7,<1,405,436,0.928899


In [11]:
metricas_por_ano_df = pd.DataFrame()
curvas_por_ano      = {}
calib_por_ano_df    = pd.DataFrame()

JANELA_ANUAL = "teste_b" if "teste_b" in JANELAS_DADOS else JANELA_PRINCIPAL
anual = JANELAS_DADOS[JANELA_ANUAL]
if COLUNA_ANO in anual["meta"].columns:
    ano_arr = pd.to_numeric(anual["meta"][COLUNA_ANO], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
    anos = sorted(int(a) for a in np.unique(ano_arr[~np.isnan(ano_arr)]))
    linhas_ano, linhas_calib = [], []

    for ano in anos:
        m = ano_arr == ano
        y_a, p_a = anual["y"][m], anual["prob"][m]
        if len(y_a) == 0:
            continue
        duas_classes = len(np.unique(y_a)) >= 2
        linhas_ano.append({
            "ano":        int(ano),
            "janela":     JANELA_ANUAL,
            "n":          int(len(y_a)),
            "positivos":  int(y_a.sum()),
            "prevalencia": float(y_a.mean()),
            "ap":   float(average_precision(y_a, p_a)) if duas_classes else float("nan"),
            "auc":  float(auc(y_a, p_a))               if duas_classes else float("nan"),
            "brier": float(brier_metric(y_a, p_a)),
            "ece":  float(ece(y_a, p_a, n_bins=N_BINS_CALIB)) if len(y_a) >= MIN_N_CALIB_ANO else float("nan"),
        })
        if len(y_a) >= MIN_N_CALIB_ANO and duas_classes:
            curvas_por_ano[int(ano)] = curva_confiabilidade(y_a, p_a, n_bins=N_BINS_CALIB)
            d = brier_decomposicao(y_a, p_a, n_bins=N_BINS_CALIB)
            linhas_calib.append({
                "ano": int(ano),
                "n":   int(len(y_a)),
                "ece": float(ece(y_a, p_a, n_bins=N_BINS_CALIB)),
                "mce": float(mce(y_a, p_a, n_bins=N_BINS_CALIB)),
                **{k: float(v) for k, v in d.items()},
            })

    metricas_por_ano_df = pd.DataFrame(linhas_ano)
    calib_por_ano_df    = pd.DataFrame(linhas_calib)

display(metricas_por_ano_df)

grupos_cfg  = list(dict.fromkeys(list(C.cfg("avaliacao", "grupos") or []) + ["regiao_residencia"]))
min_n_grupo = int(C.cfg("avaliacao", "min_n_grupo") or 100)

metricas_por_grupo_dict = {}
grupos_pulados = []
for nome_janela in dict.fromkeys([JANELA_PRINCIPAL, "teste_b"]):
    if nome_janela not in JANELAS_DADOS:
        continue
    j = JANELAS_DADOS[nome_janela]
    base_grupos = j["meta"].copy()
    if "faixa_etaria" in grupos_cfg and "faixa_etaria" not in base_grupos.columns and "idade_anos" in base_grupos.columns:
        base_grupos["faixa_etaria"] = _faixa_etaria(base_grupos["idade_anos"])
    for grupo in grupos_cfg:
        if grupo not in base_grupos.columns:
            if grupo not in grupos_pulados:
                grupos_pulados.append(grupo)
            continue
        base_grupos[grupo] = base_grupos[grupo].astype("string").fillna("AUSENTE")
        metricas_por_grupo_dict[(nome_janela, grupo)] = metricas_por_grupo(
            base_grupos, j["y"], j["prob"], grupo,
            limiar=limiar_final_val, min_n=min_n_grupo,
        )
        print(f"\n=== {nome_janela} · {grupo} ===")
        display(metricas_por_grupo_dict[(nome_janela, grupo)])
    del base_grupos

if grupos_pulados:
    print(f"\nGrupos pulados (ausentes nas previsões): {grupos_pulados}")

display(calib_por_ano_df)

,ano,janela,n,positivos,prevalencia,ap,auc,brier,ece
0,2019,teste_b,6509,781,0.119988,0.380147,0.828365,0.087094,0.013297
1,2020,teste_b,974415,272104,0.279249,0.472861,0.725946,0.187729,0.099186
2,2021,teste_b,1471091,413940,0.281383,0.453965,0.703654,0.199168,0.126205
3,2022,teste_b,470323,85051,0.180835,0.431754,0.805411,0.125749,0.049036
4,2023,teste_b,35697,3372,0.094462,0.401705,0.871131,0.068018,0.005514
5,2024,teste_b,34013,2795,0.082174,0.360502,0.866231,0.061614,0.001409
6,2025,teste_b,43026,3117,0.072445,0.341770,0.872984,0.055604,0.007648



=== teste_a · sg_uf ===


,grupo,n,prevalencia,taxa_positivos,pr_auc,pr_auc_sobre_prev,auc,roc_auc,brier,o_e,f1,precision,recall,suprimido
0,SP,31687.0,0.090416,0.090416,0.413836,4.577041,0.874018,0.874018,0.065084,1.014154,0.395630,0.259917,0.827923,False
1,MG,13816.0,0.085843,0.085843,0.261172,3.042453,0.813572,0.813572,0.071300,0.799154,0.309810,0.191191,0.816189,False
2,PR,12720.0,0.079874,0.079874,0.289536,3.624895,0.846810,0.846810,0.064221,0.803347,0.324995,0.201611,0.837598,False
3,RJ,8226.0,0.124362,0.124362,0.538758,4.332184,0.900521,0.900521,0.078321,1.395049,0.542553,0.398988,0.847507,False
4,RS,7252.0,0.099145,0.099145,0.362688,3.658154,0.855600,0.855600,0.073644,0.944276,0.375075,0.239023,0.870654,False
5,SC,6494.0,0.054820,0.054820,0.251900,4.595047,0.881359,0.881359,0.045316,0.838999,0.314502,0.198384,0.758427,False
6,BA,4800.0,0.058333,0.058333,0.416351,7.137439,0.889075,0.889075,0.042622,1.226035,0.437989,0.318699,0.700000,False
7,DF,4346.0,0.028072,0.028072,0.274755,9.787569,0.909802,0.909802,0.025440,0.481554,0.245530,0.143654,0.844262,False
8,GO,4071.0,0.098502,0.098502,0.367452,3.730419,0.858408,0.858408,0.072460,1.125290,0.416446,0.283650,0.783042,False
9,MS,3905.0,0.088604,0.088604,0.319630,3.607384,0.836640,0.836640,0.068913,1.022429,0.355820,0.230703,0.777457,False



=== teste_a · sexo_categoria ===


,grupo,n,prevalencia,taxa_positivos,pr_auc,pr_auc_sobre_prev,auc,roc_auc,brier,o_e,f1,precision,recall,suprimido
0,masculino,61480.0,0.083182,0.083182,0.390266,4.691741,0.879253,0.879253,0.060565,0.995593,0.398176,0.262080,0.828314,False
1,feminino,57746.0,0.085720,0.085720,0.345772,4.033732,0.856551,0.856551,0.065086,0.992898,0.370304,0.241129,0.797576,False
2,ignorado,19.0,0.052632,0.052632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True



=== teste_a · faixa_etaria ===


... (17 outputs left out)


=== teste_b · regiao_residencia ===


,grupo,n,prevalencia,taxa_positivos,pr_auc,pr_auc_sobre_prev,auc,roc_auc,brier,o_e,f1,precision,recall,suprimido
0,SUDESTE,1563170.0,0.249827,0.249827,0.438521,1.755298,0.729932,0.729932,0.175070,1.588754,0.497445,0.363051,0.789817,False
1,SUL,543204.0,0.231495,0.231495,0.424781,1.834945,0.738303,0.738303,0.162064,1.442295,0.479700,0.339193,0.818933,False
2,NORDESTE,472748.0,0.303853,0.303853,0.531157,1.748070,0.751683,0.751683,0.203559,1.932258,0.572689,0.449493,0.788912,False
3,CENTRO-OESTE,283888.0,0.239936,0.239936,0.431595,1.798792,0.734859,0.734859,0.170154,1.614128,0.491982,0.360382,0.774983,False
4,NORTE,171598.0,0.308652,0.308652,0.517835,1.677734,0.738988,0.738988,0.210921,2.010679,0.568143,0.448403,0.775130,False
5,AUSENTE,466.0,0.351931,0.351931,0.539487,1.532932,0.732414,0.732414,0.251777,2.654857,0.592784,0.513393,0.701220,False



Grupos pulados (ausentes nas previsões): ['cs_raca', 'cs_zona']


,ano,n,ece,mce,brier,reliability,resolution,uncertainty
0,2019,6509.0,0.013297,0.819084,0.087094,0.000435,0.018126,0.105591
1,2020,974415.0,0.099186,0.134871,0.187729,0.011128,0.023078,0.201269
2,2021,1471091.0,0.126205,0.158905,0.199168,0.017012,0.018058,0.202207
3,2022,470323.0,0.049036,0.101110,0.125749,0.003501,0.024290,0.148134
4,2023,35697.0,0.005514,0.161698,0.068018,0.000128,0.016959,0.085539
5,2024,34013.0,0.001409,0.196879,0.061614,0.000038,0.013292,0.075422
6,2025,43026.0,0.007648,0.262124,0.055604,0.000180,0.011125,0.067196


In [12]:
caminho_janelas   = TABLES_DIR / f"{prefixo}_janelas.csv"
caminho_metricas  = TABLES_DIR / f"{prefixo}_metricas.csv"
caminho_matriz    = TABLES_DIR / f"{prefixo}_matriz_confusao.csv"
caminho_por_ano   = TABLES_DIR / f"{prefixo}_metricas_por_ano.csv"
caminho_calib_ano = TABLES_DIR / f"{prefixo}_calibracao_por_ano.csv"
caminho_ic        = TABLES_DIR / f"{prefixo}_ic_bootstrap.csv"
caminho_erros     = TABLES_DIR / f"{prefixo}_analise_erros.csv"
caminho_fn        = TABLES_DIR / f"{prefixo}_fn_amostra.csv"
caminho_manifesto = TABLES_DIR / f"{prefixo}_manifesto.json"

janelas_df.to_csv(caminho_janelas, index=False, encoding="utf-8-sig")
pd.DataFrame([{
    "finalista": finalista_escolhido,
    "limiar":    limiar_final_val,
    "janela":    JANELA_PRINCIPAL,
    **{k: float(v) for k, v in metricas_test.items() if k != "limiar"},
}]).to_csv(caminho_metricas, index=False, encoding="utf-8-sig")
matriz_test.to_csv(caminho_matriz, encoding="utf-8-sig")
metricas_por_ano_df.to_csv(caminho_por_ano, index=False, encoding="utf-8-sig")
calib_por_ano_df.to_csv(caminho_calib_ano, index=False, encoding="utf-8-sig")
if not ic_df.empty:            ic_df.to_csv(caminho_ic, index=False, encoding="utf-8-sig")
if not comparacao_fn_tp.empty: comparacao_fn_tp.to_csv(caminho_erros, index=False, encoding="utf-8-sig")
if not amostra_fn.empty:       amostra_fn.to_csv(caminho_fn, index=False, encoding="utf-8-sig")

for (nome_janela, grupo), tab in metricas_por_grupo_dict.items():
    tab.to_csv(TABLES_DIR / f"{prefixo}_por_grupo_{nome_janela}_{grupo}.csv", index=False, encoding="utf-8-sig")
for grupo, tab in fn_por_grupo.items():
    tab.to_csv(TABLES_DIR / f"{prefixo}_fn_por_{grupo}.csv", index=False, encoding="utf-8-sig")
for ano, curva in curvas_por_ano.items():
    curva.to_csv(TABLES_DIR / f"{prefixo}_curva_{ano}.csv", index=False, encoding="utf-8-sig")

manifesto_c6 = {
    "cohort_kind":         manifest["cohort_kind"],
    "ensaio":              ENSAIO,
    "dataset_path":        manifest["dataset_path"],
    "dataset_sha256":      manifest["dataset_sha256"],
    "feature_columns":     FEATURES,
    "target_column":       ALVO,
    "partition_column":    PARTICAO,
    "n_treino":            int(len(treino)),
    "n_validacao":         int(len(validacao)),
    "janelas":             janelas_df.to_dict(orient="records"),
    "janela_principal":    JANELA_PRINCIPAL,
    "n_teste":             int(len(JANELAS_DADOS["teste_b"]["y"])) if "teste_b" in JANELAS_DADOS else int(len(y_test_arr)),
    "finalista_escolhido": finalista_escolhido,
    "calibracao_ativa":    calib_ativa,
    "calibracao_metodo":   calib_metodo,
    "calibracao_fonte":    calib_fonte,
    "honest_calib_split":  HONEST_CALIB_SPLIT,
    "manifesto_c5_usado":  str(manifesto_c5_path) if manifesto_c5_path else None,
    "limiar_final":        limiar_final_val,
    "limiar_metrica":      limiar_metrica,
    "teste_aberto_em_utc": abertura_teste_utc,
    "test_opened":         not ENSAIO,
    "metricas_agregadas":  {k: float(v) for k, v in metricas_test.items()},
    "ic_bootstrap":        ic_resultados,
    "erros":               {"fn": len(fn), "tp": len(tp), "fp": len(fp), "tn": len(tn)},
    "grupos_avaliados":    [f"{j}:{g}" for (j, g) in metricas_por_grupo_dict],
    "grupos_pulados":      grupos_pulados,
    "artefatos": {
        "janelas":            str(caminho_janelas),
        "metricas":           str(caminho_metricas),
        "matriz_confusao":    str(caminho_matriz),
        "metricas_por_ano":   str(caminho_por_ano),
        "calibracao_por_ano": str(caminho_calib_ano),
        "ic_bootstrap":       str(caminho_ic),
        "analise_erros":      str(caminho_erros) if not comparacao_fn_tp.empty else None,
        "fn_amostra":         str(caminho_fn)    if not amostra_fn.empty else None,
        "predicoes_teste":    str(caminho_predicoes),
    },
}
caminho_manifesto.write_text(
    json.dumps(manifesto_c6, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

with Run(
    modelo=PREFIXO_BASE,
    parametros={
        "cohort_kind":         manifest["cohort_kind"],
        "ensaio":              ENSAIO,
        "finalista":           finalista_escolhido,
        "calibracao_metodo":   calib_metodo,
        "limiar_final":        limiar_final_val,
        "limiar_metrica":      limiar_metrica,
        "teste_aberto_em_utc": abertura_teste_utc,
        "n_acessos_antes":     n_acessos_antes,
    },
    dataset={"n_treino": len(treino), "n_validacao": len(validacao), "n_teste": int(sum(len(v["y"]) for v in blocos_teste.values()))},
    features={"n_features": len(FEATURES), "usadas": FEATURES},
    notas="C6 — abertura única do teste (ensaio: metades da val_meas). Pipeline reconstruído via build_finalista.",
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(**{k: float(v) for k, v in metricas_test.items()})

print(f"Manifesto: {caminho_manifesto}")
print(f"Run registrado em: {r.pasta}")

Execução registrada em: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs\20261107T003342660471Z_11_validacao_final
Manifesto: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\tables\11_validacao_final_final_20261107T003200Z_manifesto.json
Run registrado em: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs\20261107T003342660471Z_11_validacao_final


In [13]:
def _marcar_g9_executado(root_dir: Path, timestamp_utc: str, n_antes: int) -> Path:
    yaml_path = root_dir / "configs" / "supervised.yaml"
    bruto     = yaml_path.read_bytes().decode("utf-8")
    nl        = "\r\n" if "\r\n" in bruto else "\n"
    linhas    = bruto.replace("\r\n", "\n").splitlines(keepends=True)

    i_ini = next(
        (i for i, ln in enumerate(linhas) if ln.rstrip() == "  G9_holdout:"),
        None,
    )
    if i_ini is None:
        raise RuntimeError("Bloco `G9_holdout:` não encontrado em supervised.yaml.")

    indent = len(linhas[i_ini]) - len(linhas[i_ini].lstrip(" "))
    i_fim  = len(linhas)
    for i in range(i_ini + 1, len(linhas)):
        ln = linhas[i]
        if ln.strip() and (len(ln) - len(ln.lstrip(" "))) < indent:
            i_fim = i
            break

    novos = {
        "status":       "executado",
        "executado_em": f'"{timestamp_utc}"',
        "n_acessos":    str(n_antes + 1),
    }
    padrao = re.compile(r"^(\s+)(\w+):\s*.*$")
    encontrados = set()
    bloco = linhas[i_ini:i_fim]
    for j, ln in enumerate(bloco):
        if (m := padrao.match(ln)) and m.group(2) in novos:
            bloco[j] = f"{m.group(1)}{m.group(2)}: {novos[m.group(2)]}\n"
            encontrados.add(m.group(2))

    if faltando := set(novos) - encontrados:
        raise RuntimeError(
            f"Campos ausentes em G9_holdout: {sorted(faltando)}. "
            "O teste JÁ FOI ABERTO — corrija o YAML e defina SRAG_ALLOW_TEST_REOPEN=1 para reexecutar."
        )

    linhas[i_ini:i_fim] = bloco
    yaml_path.write_bytes("".join(linhas).replace("\n", nl).encode("utf-8"))
    return yaml_path


def _registrar_teste_aberto_no_artefato(manifest_dataset_path: str, timestamp_utc: str) -> Path:
    caminho = Path(manifest_dataset_path).parent / MANIFEST_NAME
    dados = json.loads(caminho.read_text(encoding="utf-8"))
    dados["test_opened"] = True
    dados["test_opened_utc"] = timestamp_utc
    dados["test_opened_by"] = "11_validacao_final"
    caminho.write_text(json.dumps(dados, ensure_ascii=False, indent=2), encoding="utf-8")
    return caminho


if ENSAIO:
    caminho_yaml = None
    print("ENSAIO: G9 e o manifesto do artefato NÃO foram alterados.")
else:
    caminho_yaml = _marcar_g9_executado(ROOT_DIR, abertura_teste_utc, n_acessos_antes)
    caminho_manifesto_artefato = _registrar_teste_aberto_no_artefato(manifest["dataset_path"], abertura_teste_utc)
    print(f"G9 marcado em {caminho_yaml}")
    print(f"test_opened registrado em {caminho_manifesto_artefato}")

print()
print("=" * 60)
print(f"notebook 11{' (ENSAIO)' if ENSAIO else ''} concluído. AP={metricas_test['ap']:.4f}  AUC={metricas_test['auc']:.4f}  Brier={metricas_test['brier']:.4f}")
print(f"Falsos negativos: {len(fn)} de {int(y_test_arr.sum())} óbitos ({JANELA_PRINCIPAL})")
print("=" * 60)

G9 marcado em C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs\supervised.yaml
test_opened registrado em C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\analytical\manifesto_supervisionado.json

notebook 11 concluído. AP=0.3690  AUC=0.8686  Brier=0.0628
Falsos negativos: 1881 de 10065 óbitos (teste_a)
